# B-02. 특성 MLP + 임베딩 내보내기 (D·E용)

1. 특성 A+B+C → 표준화 → MLP(64-32)를 녹화 하나 빼기 21-fold로 평가
2. 마지막 은닉층(32차원)을 임베딩으로, 출력층 직전 값을 logits로 저장
   - 재질 하나 빼기 147개: `emb_mlp_{빠진재질}_{fold}.npy`, `logits_mlp_{빠진재질}_{fold}.npy`
   - 8종 21-fold: `emb_mlp_none_{fold}.npy` …
   - 모든 파일은 전체 7626구간, 데이터셋 순서. logits 열 순서는 `classes_*.json`
3. 구글 드라이브에 바로 저장 (Colab이 끊겨도 다시 실행하면 이어서 저장)

런타임은 CPU로 충분 (전체 5분 안팎)

In [ ]:
# 1. 레포 받기 + 데이터 다운로드
import os
if not os.path.exists('/content/ML-imu'):
    !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
%cd /content/ML-imu
!git pull -q
!pip install -q -r requirements.txt
!python data/download.py

In [ ]:
# 2. 드라이브 연결 (임베딩 저장 위치)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ML-imu_embeddings'   # 팀 공유 폴더가 따로 있으면 여기만 바꾸기
os.makedirs(OUT, exist_ok=True); print(OUT)

In [ ]:
# 3. 불러오기 + 특성 A+B+C
import sys, warnings
sys.path += ['src', 'notebooks/B_classic']
warnings.filterwarnings('ignore')
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from recordings import load_dataset, CLASSES
from features import extract_features
from evaluate import evaluate, log_result
from splits import load_folds, load_material_folds
from b_classic import FeatureMLP, run_loro, export_embeddings, per_recording_accuracy

X, y, rec = load_dataset('data')
F = extract_features(X, 'ABC')
print(X.shape, F.shape)

In [ ]:
# 4. 8종 분류 성능 (녹화 하나 빼기 21-fold)
pred = run_loro(FeatureMLP, F, y, rec)
m = evaluate(y, pred)
print({k: round(v, 4) for k, v in m.items()})
print('규칙 기준선: macro-F1 0.351, 정확도 0.373, 모드 macro-F1 0.666')

In [ ]:
# 5. 혼동행렬 + 녹화별 정확도
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y, pred, labels=CLASSES, normalize='true',
                                        values_format='.2f', xticks_rotation=45, ax=ax, colorbar=False)
ax.set_title('Feature MLP A+B+C (row-normalized)'); plt.tight_layout(); plt.show()
per_recording_accuracy(y, pred, rec)

In [ ]:
# 6. 임베딩 내보내기: 재질 하나 빼기 147개 (D·E용, 가장 중요)
export_embeddings(F, y, rec, load_material_folds(), OUT)

In [ ]:
# 7. 임베딩 내보내기: 8종 21-fold ('none')
export_embeddings(F, y, rec, load_folds(), OUT)

In [ ]:
# 8. 저장 확인
files = sorted(os.listdir(OUT))
print(len([f for f in files if f.startswith('emb_')]), '개 임베딩 (기대값 168 = 147 + 21)')
z = np.load(f'{OUT}/emb_mlp_carpet_0.npy'); lg = np.load(f'{OUT}/logits_mlp_carpet_0.npy')
print('emb', z.shape, z.dtype, '| logits', lg.shape, '| classes', open(f'{OUT}/classes_mlp_carpet_0.json').read())

## 결과 메모

- macro-F1 / 정확도 / 모드 macro-F1: 
- 규칙 기준선 대비: 
- 드라이브 폴더 공유 링크: 